In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/10010
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  331.6341033693067
RUN  2 , total integrated cost =  200.2000415130004
RUN  3 , total integrated cost =  118.42298957126908
RUN  4 , total integrated cost =  115.09219807860266
RUN  5 , total integrated cost =  112.17871659401163
RUN  6 , total integrated cost =  107.6741264351594
RUN  7 , total integrated cost =  104.13457068678304
RUN  8 , total integrated cost =  99.16131940364653
RUN  9 , total integrated cost =  98.70114384982266
RUN  10 , total integrated cost =  98.70063356897464
RUN  11 , total integrated cost =  98.7006297449694
RUN  12 , total integrated cost =  98.70062970088038
RUN  13 , total integrated cost =  98.7006296997742
RUN  14 , total integrated cost =  98.70062969976065
RUN  15 , total integrated cost =  98.70062969976013
RUN  16 , 

ERROR:root:Problem in initial value trasfer


RUN  19 , total integrated cost =  98.70062969975984
Control only changes marginally.
RUN  19 , total integrated cost =  98.70062969975984
Improved over  19  iterations in  13.238557828590274  seconds by  98.0636645545694  percent.
Problem in initial value trasfer:  Vmean_exc -67.76346544722767 -67.77883672190481
weight =  516.439443568426
set cost params:  1.0 0.0 516.439443568426
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4995.048846783667
Gradient descend method:  None
RUN  1 , total integrated cost =  4612.522133670742
RUN  2 , total integrated cost =  4611.234744175433
RUN  3 , total integrated cost =  4611.07359686235
RUN  4 , total integrated cost =  4610.831256941319
RUN  5 , total integrated cost =  4593.719238182297
RUN  6 , total integrated cost =  4593.719238182288


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  4593.719238182288
Control only changes marginally.
RUN  7 , total integrated cost =  4593.719238182288
Improved over  7  iterations in  0.22513259574770927  seconds by  8.03454822788764  percent.
Problem in initial value trasfer:  Vmean_exc -60.11067057889376 -60.14259501290171
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  9290.782685548957
RUN  2 , total integrated cost =  9240.798468265704
RUN  3 , total integrated cost =  9239.065184163957
RUN  4 , total integrated cost =  9238.980514236586
RUN  5 , total integrated cost =  9238.942193425675
RUN  6 , total integrated cost =  9238.916542672012
RUN  7 , total integrated cost =  9238.898907615976
RUN  8 , total integrated cost =  9238.886231401677
RUN  9 , total integrated cost =  9238.87247115258
RUN  10 , total i

...

RUN  2 , total integrated cost =  22507.657639522797
Control only changes marginally.
RUN  2 , total integrated cost =  22507.657639522797
Improved over  2  iterations in  0.2046990878880024  seconds by  1.4210854715202004e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.699093463464145 -56.699149285473815
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  32423.469731725963
RUN  2 , total integrated cost =  32298.229535541912
RUN  3 , total integrated cost =  32294.864217770424
RUN  4 , total integrated cost =  32294.752276004972


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  32294.751124297538
RUN  6 , total integrated cost =  32294.75110879189
RUN  7 , total integrated cost =  32294.75110850695
RUN  8 , total integrated cost =  32294.751108506945
RUN  9 , total integrated cost =  32294.751108506945
Control only changes marginally.
RUN  9 , total integrated cost =  32294.751108506945
Improved over  9  iterations in  0.4070165120065212  seconds by  2.989783176998259  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383213056847 -56.70383794368029
weight =  10.308192608460326
set cost params:  1.0 0.0 10.308192608460326
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32297.984316316033
Gradient descend method:  None
RUN  1 , total integrated cost =  32297.984316299673
RUN  2 , total integrated cost =  32297.98431629916
RUN  3 , total integrated cost =  32297.984316299135
RUN  4 , total integrated cost =  32297.98431629913


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  32297.98431629913
Control only changes marginally.
RUN  5 , total integrated cost =  32297.98431629913
Improved over  5  iterations in  0.20569820515811443  seconds by  5.233857791608898e-11  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383213056775 -56.70383794367966


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
[0, 5] []
closest index  5
set 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1286 , total integrated cost =  4353.013158182087
Improved over  1286  iterations in  30.890300557017326  seconds by  49.572346660661346  percent.
Problem in initial value trasfer:  Vmean_exc -56.627833537591634 -56.62781017878954
weight =  20.93137824103441
set cost params:  1.0 0.0 20.93137824103441
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4739.447520143562
Gradient descend method:  None
RUN  1 , total integrated cost =  4739.447520143562
Control only changes marginally.
RUN  1 , total integrated cost =  4739.447520143562
Improved over  1  iterations in  0.070121755823493  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.627833537591634 -56.62781017878954
-------  15 0.4500000000000001 0.4500000000000002
[0, 5] []
closest index  5
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12695.857361224913
Gradient descend m

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  2000 , total integrated cost =  9246.537457519818
RUN  2000 , total integrated cost =  9246.537457519818
Improved over  2000  iterations in  46.356748037040234  seconds by  27.16886150784778  percent.
Problem in initial value trasfer:  Vmean_exc -56.64473404060383 -56.6451767584508
weight =  14.078864331814723
set cost params:  1.0 0.0 14.078864331814723
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9347.651458197073
Gradient descend method:  None
RUN  1 , total integrated cost =  9347.651458197073
Control only changes marginally.
RUN  1 , total integrated cost =  9347.651458197073
Improved over  1  iterations in  0.06945907883346081  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.64473404060383 -56.6451767584508
-------  20 0.4500000000000001 0.4750000000000002
[0, 5] []
closest index  5
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12444.2309483674

...

Control only changes marginally.
RUN  27 , total integrated cost =  22497.205239017898
Improved over  27  iterations in  1.0445416364818811  seconds by  4.087952272579116  percent.
Problem in initial value trasfer:  Vmean_exc -56.699104167490326 -56.69916070019743
weight =  10.460248681147423
set cost params:  1.0 0.0 10.460248681147423
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22502.12412328109
Gradient descend method:  None
RUN  1 , total integrated cost =  22502.12412328109
Control only changes marginally.
RUN  1 , total integrated cost =  22502.12412328109
Improved over  1  iterations in  0.12073959223926067  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.699104167490326 -56.69916070019743
-------  140 0.4500000000000001 0.9000000000000006
[0, 5] [5]
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9902.376663195739
Gradient descend 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  264 , total integrated cost =  9091.769361915658
Improved over  264  iterations in  8.161876453086734  seconds by  8.185987352843  percent.
Problem in initial value trasfer:  Vmean_exc -56.64432233784096 -56.64449521408107
weight =  11.020922462633873
set cost params:  1.0 0.0 11.020922462633873
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9105.14911070693
Gradient descend method:  None
RUN  1 , total integrated cost =  9105.149110706929
RUN  2 , total integrated cost =  9105.149110706929
Control only changes marginally.
RUN  2 , total integrated cost =  9105.149110706929
Improved over  2  iterations in  0.12101813219487667  seconds by  2.842170943040401e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.64432233784096 -56.64449521408107
-------  145 0.5750000000000002 0.9000000000000006
[0, 5] [5]
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  32280.451756427577
RUN  13 , total integrated cost =  32280.451756427574
RUN  14 , total integrated cost =  32280.45175642757
RUN  15 , total integrated cost =  32280.45175642757
Control only changes marginally.
RUN  15 , total integrated cost =  32280.45175642757
Improved over  15  iterations in  0.3562755733728409  seconds by  2.8311476736362664  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383247039459 -56.7038360685859
weight =  10.312758854203185
set cost params:  1.0 0.0 10.312758854203185
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32283.273093029602
Gradient descend method:  None
RUN  1 , total integrated cost =  32283.273093029602
Control only changes marginally.
RUN  1 , total integrated cost =  32283.273093029602
Improved over  1  iterations in  0.07164505496621132  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383247039459 -56.7038360685859
--------------------------

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  572.0523321281835
set cost params:  1.0 0.0 572.0523321281835
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5067.930155163365
Gradient descend method:  None
RUN  1 , total integrated cost =  5067.881064932709
RUN  2 , total integrated cost =  5067.880959521125
RUN  3 , total integrated cost =  5067.880958442712


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  5067.880958425934
RUN  5 , total integrated cost =  5067.880958425594
RUN  6 , total integrated cost =  5067.8809584255905
RUN  7 , total integrated cost =  5067.880958425589
RUN  8 , total integrated cost =  5067.880958425589
Control only changes marginally.
RUN  8 , total integrated cost =  5067.880958425589
Improved over  8  iterations in  0.3902619332075119  seconds by  0.0009707461679653306  percent.
Problem in initial value trasfer:  Vmean_exc -60.03638701934224 -60.06799238500771
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
no convergence
weight =  11.546009345962455
set cost params:  1.0 0.0 11.5460093459

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  27289.28328467494
Control only changes marginally.
RUN  3 , total integrated cost =  27289.28328467494
Improved over  3  iterations in  0.2517472766339779  seconds by  6.449880629588733e-10  percent.
Problem in initial value trasfer:  Vmean_exc -56.70355632314341 -56.703631783272755
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
no convergence
weight =  10.382440553704248
set cost params:  1.0 0.0 10.382440553704248
interpolate adjoint :  True True True
RUN  0 , total integrated co

...

RUN  4 , total integrated cost =  27389.98314366641
RUN  5 , total integrated cost =  27389.98314366639
RUN  6 , total integrated cost =  27389.98314366638
RUN  7 , total integrated cost =  27389.98314366638
Control only changes marginally.
RUN  7 , total integrated cost =  27389.98314366638
Improved over  7  iterations in  0.3807971552014351  seconds by  2.340713933790539e-08  percent.
Problem in initial value trasfer:  Vmean_exc -56.7035563240121 -56.70363178415312
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
c

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  39 , total integrated cost =  27407.77222748833
Improved over  39  iterations in  1.5110208820551634  seconds by  0.00028375976943095793  percent.
Problem in initial value trasfer:  Vmean_exc -56.703556386267486 -56.70363185851754
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  27427.44843166243
Control only changes marginally.
RUN  1 , total integrated cost =  27427.44843166243
Improved over  1  iterations in  0.07513966038823128  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.703556386267486 -56.70363185851754
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.600

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  280.7050122619814
Gradient descend method:  None
RUN  1 , total integrated cost =  86.10778307939776
RUN  2 , total integrated cost =  81.87181947798236
RUN  3 , total integrated cost =  77.89829945077364
RUN  4 , total integrated cost =  15.398183580230999
RUN  5 , total integrated cost =  6.811649789764248
RUN  6 , total integrated cost =  6.801702202048697
RUN  7 , total integrated cost =  6.800971763130008
RUN  8 , total integrated cost =  6.800827331911587
RUN  9 , total integrated cost =  6.800451488719094
RUN  10 , total integrated cost =  6.7768206306603425
RUN  11 , total integrated cost =  6.763958576312832
RUN  12 , total integrated cost =  6.763908452645437
RUN  13 , total integrated cost =  6.763887828109977
RUN  14 , total integrated cost =  6.763868831617274
RUN  15 , total integrated cost =  6.763821881724817
RUN  16 , 

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  6.711665476387969
Control only changes marginally.
RUN  60 , total integrated cost =  6.711665476387969
Improved over  60  iterations in  7.664277048781514  seconds by  97.60899692445678  percent.
Problem in initial value trasfer:  Vmean_exc -67.78096066651307 -67.78500809327385
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  66.57237762600326
Gradient descend method:  HS
RUN  1 , total integrated cost =  65.76696917449416
RUN  2 , total integrated cost =  65.58031796421065
RUN  3 , total integrated cost =  65.55027954338253


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  65.5502795433825
RUN  5 , total integrated cost =  65.5502795433825
Control only changes marginally.
RUN  5 , total integrated cost =  65.5502795433825
Improved over  5  iterations in  0.7978896740823984  seconds by  1.5353185796709852  percent.
Problem in initial value trasfer:  Vmean_exc -67.56694966086371 -67.5745935394942
weight =  7775.152693332503
set cost params:  1.0 0.0 7775.152693332503
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5054.185703316875
Gradient descend method:  None
RUN  1 , total integrated cost =  4894.534877964522
RUN  2 , total integrated cost =  4894.533938103704
RUN  3 , total integrated cost =  4894.533928433159
RUN  4 , total integrated cost =  4894.533928372254
RUN  5 , total integrated cost =  4894.533928372241
RUN  6 , total integrated cost =  4894.533928372238


...

RUN  50 , total integrated cost =  57.5842966384013
Control only changes marginally.
RUN  50 , total integrated cost =  57.5842966384013
Improved over  50  iterations in  4.508125972002745  seconds by  99.82161248215203  percent.
Problem in initial value trasfer:  Vmean_exc -64.18007917366238 -64.19695845661876
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  557.6758627204869
Gradient descend method:  HS
RUN  1 , total integrated cost =  538.9268575058651
RUN  2 , total integrated cost =  535.7426598177801
RUN  3 , total integrated cost =  535.6775255107331
RUN  4 , total integrated cost =  535.6403474885001


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  535.6403474885001
Control only changes marginally.
RUN  5 , total integrated cost =  535.6403474885001
Improved over  5  iterations in  1.3166976291686296  seconds by  3.951312349164951  percent.
Problem in initial value trasfer:  Vmean_exc -61.36922404465158 -61.37802004037849
weight =  6214.000722586985
set cost params:  1.0 0.0 6214.000722586985
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32380.214264760823
Gradient descend method:  None
RUN  1 , total integrated cost =  30930.593365625286
RUN  2 , total integrated cost =  30918.870198807865
RUN  3 , total integrated cost =  30910.9796932705
RUN  4 , total integrated cost =  30893.90542894665
RUN  5 , total integrated cost =  30881.011460705533
RUN  6 , total integrated cost =  30805.472529401428
RUN  7 , total integrated cost =  30756.437340833778
RUN  8 , total integrated cost =  30718.942767527195
RUN  9 , total integrated cost =  30690.68132701998
RUN  10 , total inte

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  28841.01720495174
Control only changes marginally.
RUN  61 , total integrated cost =  28841.01720495174
Improved over  61  iterations in  11.741386003792286  seconds by  10.930122422509015  percent.
Problem in initial value trasfer:  Vmean_exc -56.682338573986925 -56.6851311246165


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8096.238126532678
set cost params:  1.0 0.0 8096.238126532678
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.386141054696
Gradient descend method:  None
RUN  1 , total integrated cost =  5094.345754997623
RUN  2 , total integrated cost =  5094.34475720695
RUN  3 , total integrated cost =  5094.344751722653
RUN  4 , total integrated cost =  5094.344751722639


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  5094.344751722639
Control only changes marginally.
RUN  5 , total integrated cost =  5094.344751722639
Improved over  5  iterations in  1.4634873755276203  seconds by  0.0008124498400974289  percent.
Problem in initial value trasfer:  Vmean_exc -63.373176852527465 -63.42022213206527
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5804.823983144613
set cost params:  1.0 0.0 5804.823983144613
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13012.567634466133
Gradient descend method:  None
RUN  1 , total integrated cost =  13012.531407620942
RUN  2 , total integrated cost =  13012.531407014732
RUN  3 , total integrated cost =  13012.531407012704
RUN  4 , total integrated cost =  13012.531407012673
RUN  5 , total integrated cost =  13012.531407012668
RUN  6 , total integrated cost =  13012.53140701266


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  13012.53140701266
Control only changes marginally.
RUN  7 , total integrated cost =  13012.53140701266
Improved over  7  iterations in  1.1482023261487484  seconds by  0.0002784035748391034  percent.
Problem in initial value trasfer:  Vmean_exc -59.78810289855062 -59.80542631686921
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6513.09515050305
set cost params:  1.0 0.0 6513.09515050305
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8229.6132516774
Gradient descend method:  None
RUN  1 , total integrated cost =  8229.602925193281
RUN  2 , total integrated cost =  8229.602858152259
RUN  3 , total integrated cost =  8229.602857243279
RUN  4 , total integrated cost =  8229.602857224698
RUN  5 , total integrated cost =  8229.602857223837
RUN  6 , total integrated cost =  8229.602857223828
RUN  7 , total integrated cost =  8229.602857223825


...

RUN  4 , total integrated cost =  34490.91911364338
Control only changes marginally.
RUN  4 , total integrated cost =  34490.91911364338
Improved over  4  iterations in  0.7121393028646708  seconds by  9.23872676139581e-08  percent.
Problem in initial value trasfer:  Vmean_exc -56.70346151859648 -56.703430432654805
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 99
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True

ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  34490.952307452586
Control only changes marginally.
RUN  5 , total integrated cost =  34490.952307452586
Improved over  5  iterations in  0.8321342281997204  seconds by  1.0413032214273699e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.703460585794566 -56.703429587507664
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 100
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True],

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  34490.984247533386
RUN  4 , total integrated cost =  34490.984247533386
Control only changes marginally.
RUN  4 , total integrated cost =  34490.984247533386
Improved over  4  iterations in  0.7210226338356733  seconds by  2.142532053994728e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.7034595839983 -56.703428679863634
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  58.34432528331472
Gradient descend method:  None
RUN  1 , total integrated cost =  1.067864461919698
RUN  2 , total integrated cost =  1.0632557653726646
RUN  3 , total integrated cost =  1.0632557653726575
RUN  4 , total integrated cost =  1.0632557653726569
RUN  5 , total integrated cost =  1.0632557653726564


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  1.0632557653726564
Control only changes marginally.
RUN  6 , total integrated cost =  1.0632557653726564
Improved over  6  iterations in  0.3894516173750162  seconds by  98.17761922824613  percent.
Problem in initial value trasfer:  Vmean_exc -67.79656912909738 -67.80047699708481
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  89.96034554019012
Gradient descend method:  None
RUN  1 , total integrated cost =  4.069165337250347
RUN  2 , total integrated cost =  4.068538906232759
RUN  3 , total integrated cost =  4.068363049808582
RUN  4 , total integrated cost =  4.068246642264736
RUN  5 , total integrated cost =  4.068082851694852
RUN  6 , total integrated cost =  4.067755982122818
RUN  7 , total integrated cost =  4.064896498911823
RUN  8 , total integrated cost =  4.058882659296786
RUN  9 , total integrated cost =  4.0588826592967715
RUN  10 , total

ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  4.058882659296762
Control only changes marginally.
RUN  12 , total integrated cost =  4.058882659296762
Improved over  12  iterations in  0.48061726056039333  seconds by  95.48814243106321  percent.
Problem in initial value trasfer:  Vmean_exc -66.96292408790612 -66.9740432310256
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  47.23395524872358
Gradient descend method:  None
RUN  1 , total integrated cost =  1.9753085615953143
RUN  2 , total integrated cost =  1.9745485234253572
RUN  3 , total integrated cost =  1.9745485234253544
RUN  4 , total integrated cost =  1.974548523425324
RUN  5 , total integrated cost =  1.9745485234253226


...

RUN  13 , total integrated cost =  9.53588277821047
RUN  14 , total integrated cost =  9.53588277821047
Control only changes marginally.
RUN  14 , total integrated cost =  9.53588277821047
Improved over  14  iterations in  0.5952156353741884  seconds by  92.24153687789311  percent.
Problem in initial value trasfer:  Vmean_exc -69.36850134630343 -69.4108204924754
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  299.39117749997365
Gradient descend method:  None
RUN  1 , total integrated cost =  41.36047188767223
RUN  2 , total integrated cost =  38.72913747950982
RUN  3 , total integrated cost =  37.992830711218254
RUN  4 , total integrated cost =  37.70431782046984
RUN  5 , total integrated cost =  37.54600656428467
RUN  6 , total integrated cost =  37.45927355220869
RUN  7 , total integrated cost =  37.41705679445081
RUN  8 , total integrated cost =  37.390262373933496
RUN  9 , total

ERROR:root:Problem in initial value trasfer


RUN  50 , total integrated cost =  37.30182824629822
Control only changes marginally.
RUN  53 , total integrated cost =  37.301828238507795
Improved over  53  iterations in  2.028762273490429  seconds by  87.54077239349812  percent.
Problem in initial value trasfer:  Vmean_exc -63.21173715159895 -63.243260233911215
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32041.582201458463
Gradient descend method:  None
RUN  1 , total integrated cost =  30972.058955314096
RUN  2 , total integrated cost =  30957.82244429766
RUN  3 , total integrated cost =  30957.711908456324
RUN  4 , total integrated cost =  30957.707793794423
RUN  5 , total integrated cost =  30957.7077371842
RUN  6 , total integrated cost =  30957.707494679802
RUN  7 , total integrated cost =  30957.7072238541
RUN  8 , total integrated cost =  30957.707020077396
RUN  9 , total integrated cost =  30957.70680903914
RUN  10 ,

ERROR:root:Problem in initial value trasfer


RUN  19 , total integrated cost =  30957.705994980483
RUN  20 , total integrated cost =  30957.705993246556
Control only changes marginally.
RUN  22 , total integrated cost =  30957.70599324654
Improved over  22  iterations in  0.8354752995073795  seconds by  3.3827174993954685  percent.
Problem in initial value trasfer:  Vmean_exc -56.70400377444256 -56.70400222417645


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.0632557653726564
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.0632557653726564
Control only changes marginally.
RUN  1 , total integrated cost =  1.0632557653726564
Improved over  1  iterations in  0.063865527510643  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.79656912909738 -67.80047699708481
-------  15 0.4500000000000001 0.4500000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4.058882659296762
Gradient descend method:  None
RUN  1 , total integrated cost =  4.058882659296762
Control only changes marginally.
RUN  1 , total integrated cost =  4.058882659296762
Improved over  1  iterations in  0.06395029090344906  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.96292408790612 -66.9740432310256
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.9745485234253226

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.9745485234253226
Control only changes marginally.
RUN  1 , total integrated cost =  1.9745485234253226
Improved over  1  iterations in  0.06403274089097977  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -70.3079006859571 -70.33125887832702
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13.003848035607488
Gradient descend method:  None


...

RUN  1 , total integrated cost =  1.9348085734132883
Control only changes marginally.
RUN  1 , total integrated cost =  1.9348085734132883
Improved over  1  iterations in  0.06414429657161236  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.78449198395883 -73.82818156400512
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9.53588277821047
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  9.53588277821047
Control only changes marginally.
RUN  1 , total integrated cost =  9.53588277821047
Improved over  1  iterations in  0.06412741728127003  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -69.36850134630343 -69.4108204924754
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  37.301828238507795
Gradient descend method:  None
RUN  1 , total integrated cost =  37.301828238507795
Control only changes marginally.
RUN  1 , total integrated cost =  37.301828238507795
Improved over  1  iterations in  0.06577493622899055  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -63.21173715159895 -63.243260233911215
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30957.70599324

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  30957.70599324654
Control only changes marginally.
RUN  1 , total integrated cost =  30957.70599324654
Improved over  1  iterations in  0.06902316398918629  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70400377444256 -56.70400222417645
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
